In [1]:
!pip install --upgrade --quiet llama-cpp-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.3/50.3 MB 24.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 826.5 kB/s eta 0:00:00


In [19]:
from llama_cpp import Llama
from transformers import AutoTokenizer, BertTokenizer
import json
import csv
import os
import random

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Text Generation  
---



Aufgabe:  
Das Modell erhält Informationen über aktuellen Redner, den Text der bisherigen Rede und die Parteizugehörigkeit des Störers. Jetzt soll ein Zwischenruf formuliert werden.  
<br>
Ergebnisse:  
Die Ergebnisse sind in [text_generation_results](text_generation_results.json) gespeichert.  
Zu jedem Prompt wird zusätzlich der preceding_context und Unterbrecher gespeichert, sowie die tatsächliche Unterbrechung und alle generierten Unterbrechungen mit verschieden Konfigurationen.  
Der erste Prompt hat noch nicht die tatsächliche Unterbrechung gespeichert.

## Vorbereitung
---

Ich hole mir zuerst alle [parsed_protocols](../_data/parsed_protocols)

In [53]:
def get_parsed_protocols():
  folder_path = "/content/drive/MyDrive/parsed_protocols" # set path to parsed_protocols
  json_data_list = []

  for index, filename in enumerate(os.listdir(folder_path)):
      if filename.endswith('.json'):
          file_path = os.path.join(folder_path, filename)
          with open(file_path, 'r') as file:
              data = json.load(file)
              json_data_list.append(data)
              print(f"Successfully read {filename}")
  return json_data_list

Ich schreibe mir eine Klasse, die aus allen Protokollen aus [parsed_protocols](../_data/parsed_protocols) eine zufällige Unterbrechung auswählt und diese zurückgibt mit weiteren Informationen zum Redner, Unterbrecher, zur tatsächlichen Unterbrechung und dem vorangehenden Kontext.

In [91]:
class ProtocolProcessor:
    def __init__(self, parsed_protocols):
        self.parsed_protocols = parsed_protocols
        self.processed_indices = set()
        self.current_inner_index = None
        self.remaining_inner_indices = list(range(len(parsed_protocols)))
        self.used_comments = set()

    def get_random_protocol_with_comments(self):
        # If there are no remaining inner indices to process, return None
        if not self.remaining_inner_indices:
            return None

        # Select a random inner index if not already selected
        if self.current_inner_index is None or self.current_inner_index not in self.remaining_inner_indices:
            self.current_inner_index = random.choice(self.remaining_inner_indices)

        inner_list = self.parsed_protocols[self.current_inner_index]

        # Find protocols with non-empty comments
        protocols_with_comments = [protocol for protocol in inner_list if 'comments' in protocol and protocol['comments']]

        # If no protocols with comments in current inner list, remove from remaining indices and retry
        if not protocols_with_comments:
            self.remaining_inner_indices.remove(self.current_inner_index)
            return self.get_random_protocol_with_comments()

        # Randomly select a protocol with comments
        protocol = random.choice(protocols_with_comments)

        # Filter out used comments
        comments = [comment for comment in protocol['comments']
                    if (self.current_inner_index, inner_list.index(protocol), protocol['comments'].index(comment)) not in self.used_comments]

        # If all comments are used, mark protocol as processed and retry
        if not comments:
            self.processed_indices.add((self.current_inner_index, inner_list.index(protocol)))
            self.remaining_inner_indices.remove(self.current_inner_index)
            return self.get_random_protocol_with_comments()

        # Select a random comment from remaining comments
        selected_comment = random.choice(comments)
        self.used_comments.add((self.current_inner_index, inner_list.index(protocol), protocol['comments'].index(selected_comment)))

        # Extract information
        speaker_info = protocol.get('speaker', {})
        commentator_info = selected_comment.get('commentator', {})
        text = selected_comment.get('text', '')
        preceding_context = selected_comment.get('preceding_context', '')

        return {
            'speaker_info': speaker_info,
            'commentator_info': commentator_info,
            'text': text,
            'preceding_context': preceding_context
        }

Und eine zusätzliche Methode, die Exception handling beinhaltet und den preceding_context vorbereitet. Es wird dafür Redner und Unterbrecher Informationen am Ende des Kontexts hinzugefügt für den System Prompt.

In [90]:
def get_interruption_info(parsed_protocols):
  # Fetch random protocol with a random comment
  comment_data = processor.get_random_protocol_with_comments()

  if comment_data:
    preceding_context = comment_data['preceding_context']
    prompt_info = f"{{{comment_data['speaker_info']['name']}, {comment_data['speaker_info']['party']}, {comment_data['commentator_info']['name']}, {comment_data['commentator_info']['party']}}}"
    preceding_context += prompt_info
    return preceding_context, comment_data['commentator_info'], comment_data['text']
  else:
    preceding_context = ""
    print("No protocols with comments found or all have been processed.")
    raise Exception("No protocols with comments found or all have been processed.")

---

Ich schreibe mir zusätzlich eine configurations.json Datei, die Kombinationen aus n_ctx, temperature, top_p und top_k speichert.  
Ursrpünglich waren mehr Kombinationen geplant:  
> n_ctx: 2048, 4096, 8192  
> temperature: 0.4, 0.7, 1.0  
> top_p: 0.85, 0.95, 1.0  
> top_k: 15, 40, 50
  
Aus zeitlichen Gründen wurden die Parameter reduziert und sich auf das Prompting (Few-Shot-Learning) konzentriert.  
Neue Parameter:  
> n_ctx: 8192  
> temperature: 0.8, 0.95  
> top_p: 0.85, 0.95  
> top_k: 40  

In [77]:
import itertools

def write_configurations():
  # Define possible values for each parameter

  # reduced due to time constraints
  # n_ctx_values = [2048, 4096, 8192]
  # temperature_values = [0.4, 0.7, 1.0]
  # top_p_values = [0.85, 0.95, 1.0]
  # top_k_values = [15, 40, 50]

  n_ctx_values = [8192]
  temperature_values = [0.8, 0.95]
  top_p_values = [0.85, 0.95]
  top_k_values = [40]

  # Generate all combinations of parameters
  combinations = list(itertools.product(n_ctx_values, temperature_values, top_p_values, top_k_values))

  # Create configurations list
  configurations = []
  for combo in combinations:
      n_ctx, temperature, top_p, top_k = combo
      config = {
          "n_ctx": n_ctx,
          "max_tokens": max_tokens,
          "temperature": temperature,
          "top_p": top_p,
          "top_k": top_k,
      }
      configurations.append(config)

  # Save configurations to a JSON file
  with open('configurations.json', 'w') as f:
      json.dump(configurations, f, indent=4)


---

Ich brauche eine Methode, die ein neues LLM mit den neuen Parametern initiiert:

In [155]:
def setup_llm(new_n=n, new_n_ctx=n_ctx, new_max_tokens=max_tokens, new_temperature=temperature, new_top_p=top_p, new_top_k=top_k):
  global n, n_ctx, max_tokens, temperature, top_p, top_k

  # Number of responses per configuration
  n = new_n

  n_ctx = new_n_ctx
  max_tokens = new_max_tokens
  temperature = new_temperature
  top_p = new_top_p
  top_k = new_top_k

  # Build the Llama model
  llm = Llama.from_pretrained(
    repo_id=model_name,
    filename=quant_type,
    n_ctx=n_ctx,
    n_threads=n_threads,
    n_gpu_layers=n_gpu_layers,
    verbose=True
  )

  print(f"Model setup with n_ctx: {n_ctx}, temperature: {temperature}, top_p: {top_p}, top_k: {top_k}")
  return llm

---

Da der preceding_context oft sehr lang ist und dieser die n_ctx des Modells überschreitet, muss ich die Rede kürzen. Ich rechne mir also die Tokenanzahl aus, die der Prompt benötigt und reduziere meinen preceding_context entsprechend.  
Da das LLM einen anderen Tokenizer benutzt als ich (bert-base-german-cased), muss ich den preceding_context nochmal weiter reduzieren. (bert-base-german-cased generiert mehr Token als der LLM-Tokenizer)

In [143]:
 # specific to this prompt template

def truncate_speech(prompt, preceding_context, n_ctx_trunc):
  # Calculate the amount of tokens of the prompt template without the preceding_context
  overhead = prompt.replace(preceding_context, "", 1)
  overhead_tokens = tokenizer.encode(overhead, add_special_tokens=False)
  overhead_length = len(overhead_tokens)

  tokens = tokenizer.encode(preceding_context, add_special_tokens=False)
  # Cut off from the start of the speech
  tokens = tokens[-(round(n_ctx-n_ctx_trunc)-overhead_length):]
  tokenized_speech = tokenizer.decode(tokens, skip_special_tokens=True)

  # Generate the new prompt by replacing preceding_context with the truncated_context
  new_prompt = prompt.replace(preceding_context, tokenized_speech, 1)

  return new_prompt, tokenized_speech

---

Um die n Antworten zu generieren rufe ich create_completion vom LLM auf.

In [163]:
def generate_responses(prompt):
  responses = []
  for i in range(n):
    # Generate text using the Llama model
    responses.append(
        llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k
        )['choices'][0]['text'].strip()
    )

    print(responses[i])

  return responses

In [11]:
def print_response(response):
  for choice in response['choices']:
    print(choice['text'])

---

Ich speichere meine Ergebnisse als JSON Datei ab. Dafür muss ich die Informationen erst strukturieren.

In [99]:
def append_interruption(responses):
  interruption = {
      "n_ctx": n_ctx,
      "max_tokens": max_tokens,
      "temperature": temperature,
      "top_p": top_p,
      "top_k": top_k,
      "responses": responses
  }
  interruptions.append(interruption)

In [98]:
def save_interruptions(prompt, preceding_context, interruptor, interruptions, actual_interruption):
  json_file = 'text_generation_results.json'

  try:
      with open(json_file, 'r') as f:
          data = json.load(f)
  except FileNotFoundError:
      data = []

  data.append(
      {
          "prompt": prompt,
          "preceding_context": preceding_context,
          "interruptor": interruptor,
          "actual_interruption": actual_interruption,
          "interruptions": interruptions
      }
      )

  with open(json_file, 'w') as f:
    json.dump(data, f, indent=4)

  # empty interruptions
  interruptions.clear()

  print(f"Data saved to {json_file}")

## Initialisierung
---

Zuerst initiiere ich alle Attribute wie den model_name, quant_type, max_tokens, temperature, tokenizer, processor, etc.

In [171]:
# Define attributes
model_name = "bartowski/Llama-3-SauerkrautLM-8b-Instruct-GGUF"
quant_type = "*Q4_K_M.gguf"
n_threads = 2
n_gpu_layers = 2

# Number of responses per configuration
n = 2

# Default values
n_ctx = 2048
max_tokens=50
temperature = 0.8
top_p = 0.95
top_k = 40

In [172]:
tokenizer = BertTokenizer.from_pretrained('bert-base-german-cased')

In [173]:
parsed_protocols = get_parsed_protocols()

Successfully read 20_083_2023-01-27.json
Successfully read 20_082_2023-01-26.json
Successfully read 20_081_2023-01-25.json
Successfully read 16_059_2006-10-25.json
Successfully read 16_060_2006-10-26.json
Successfully read 16_063_2006-11-09.json
Successfully read 16_061_2006-10-27.json
Successfully read 16_062_2006-11-08.json
Successfully read 16_065_2006-11-21.json
Successfully read 16_064_2006-11-10.json
Successfully read 16_066_2006-11-22.json
Successfully read 16_068_2006-11-24.json
Successfully read 16_067_2006-11-23.json
Successfully read 16_069_2006-11-29.json
Successfully read 16_070_2006-11-30.json
Successfully read 16_072_2006-12-13.json
Successfully read 16_073_2006-12-14.json
Successfully read 16_071_2006-12-01.json
Successfully read 16_075_2007-01-17.json
Successfully read 16_074_2006-12-15.json
Successfully read 16_078_2007-01-31.json
Successfully read 16_076_2007-01-18.json


KeyboardInterrupt: 

In [ ]:
processor = ProtocolProcessor(parsed_protocols)

In [ ]:
write_configurations()

In [ ]:
interruptions = []

## System Prompts
---

Der erste Prompt scheinte schon angemessen zu funktionieren. Nach kleinen Änderungen habe ich meinen System Prompt finalisiert auf meine zweite Version.

Hier wäre es natürlich möglich, viele verschiedene Prompts auszuprobieren.  
Aus zeitlichen Gründen wurde dieser Schritt aber leider ausgelassen.

Erster Prompt:  

    "Dir wird eine Rede im Bundestag übergeben. An der Stelle, an der die Rede aufhört, sollst du eine Unterbrechung generieren. "
    "Die Unterbrechung kann eine Zustimmung, Widerruf, Frage oder eine Aussage sein. "
    "Es wird in den Klammern \"{Störer, Partei}\" am Ende zusätzlich mitgegeben, wer du bist und welcher Partei du angehörst. "
    "Packe deine Antwort in dieses Muster: (Beifall bei {Artikel} {Partei}:)"


Zweiter Prompt:<br>

    "Dir wird eine Rede im Bundestag übergeben. An der Stelle, an der die Rede aufhört, sollst du eine Unterbrechung generieren. "
    "Die Unterbrechung kann eine Zustimmung, Widerruf, Frage oder eine Aussage sein. "
    "Es wird in den Klammern \"{Redner, Redner_Partei, Störer, Störer_Partei}\" am Ende zusätzlich mitgegeben, wer der Redner ist, wer du bist und welcher Partei beide jeweils angehören. "
    "Packe deine Antwort in dieses Muster: (Beifall bei {Artikel} {Störer_Partei}: {Deine Antwort})"

In [67]:
system_prompt = (
    "Dir wird eine Rede im Bundestag übergeben. An der Stelle, an der die Rede aufhört, sollst du eine Unterbrechung generieren. "
    "Die Unterbrechung kann eine Zustimmung, Widerruf, Frage oder eine Aussage sein. "
    "Es wird in den Klammern \"{Redner, Redner_Partei, Störer, Störer_Partei}\" am Ende zusätzlich mitgegeben, wer der Redner ist, wer du bist und welcher Partei beide jeweils angehören. "
    "Packe deine Antwort in dieses Muster: (Beifall bei {Artikel} {Störer_Partei}: {Deine Antwort})"
)


## Beispiel-Antworten
---

Für One-Shot-Learning und Few-Shot-Learning habe ich mir Beispiele aus den [parsed_protocols](../_data/parsed_protocols) ausgesucht. Die Beispiele dienten dazu, die Antwortstruktur einzuhalten, und dann zusätzlich hoffentlich häufiger kurze Unterbrechungen zu generieren.

In [134]:
# Zuerst gewöhnliche Antworten

# Eine positive Unterbrechung
example_context_1 = "Sie schaffen dieses Recht nur, weil Sie damit Personen mit ungekl\u00e4rter Identit\u00e4t die Br\u00fccke in ein Daueraufenthaltsrecht bauen wollen.{Alexander Throm, CDU/CSU, Andrea Lindholz, CDU/CSU}"
example_response_1 = "(Beifall bei der CDU/CSU: Genau! So ist es!)"

# Negative Unterbrechungen
example_context_2 = "Wir brauchen vor allem aber auch den Dialog mit den politisch Verantwortlichen. Darum w\u00e4re es ein Fehler, auf die Beteiligung des Iran an der Islam-Konferenz zu verzichten. Das w\u00e4re, wie wenn man versuchen wollte, die k\u00fcnftige Strategie der NATO ohne die USA zu beschlie\u00dfen.{Dr. Andreas Schockenhoff, CDU/CSU, Freimut Duve, SPD}"
example_response_2 = "(Beifall bei der SPD: Na, na! Man muß als Politiker mit dem Vergleich ein bißchen vorsichtig sein.)"

example_context_3 = "(Dr. Frank-Walter Steinmeier [SPD]: Das ist doch Quatsch! Jetzt hören Sie doch mit diesem Unsinn auf!) – Hätten Sie diesen Unsinn nicht gesagt, dann bräuchte ich ihn nicht zu zitieren.{Daniela Ludwig, CDU/CSU, Caren Marks, SPD}"
example_response_3 = "(Zuruf bei der SPD: Ja, dann zitieren Sie mal richtig!)"

example_context_4 = "Die Grünen wiederum wollen mit diesem Gesetz weitere Ideologen Ihrer Partei in gutbezahlte Jobs und Posten bringen, die dann, wie damals zu DDR-Zeiten, die arbeitende Bevölkerung und in diesem Fall unsere deutschen Landwirte denunzieren sollen.{Frank Rinck, AfD, Karl Bär, GRÜNE}"
example_response_4 = "(Beifall bei der GRÜNE: So ein Quatsch!)"

# Und dann eine neutrale Unterbrechung
example_context_5 = "Aber nicht umsonst klagen kinderreiche Familien vor dem Bundesverfassungsgericht gegen die Ökosteuer.  Wir fordern als einzige Partei die Abschaffung der Ökosteuer, weil sie vom Konzept her falsch ist.{Rainer Brüderle, FDP, Joachim Poß, SPD}"
example_response_5 = "(Beifall bei der SPD: Dann steigt aber der Rentenversicherungsbeitrag!)"

Ich musste meine Antworten auch in meine Beispielstruktur packen:

example_context_1 = "Sie schaffen dieses Recht nur, weil Sie damit Personen mit ungekl\u00e4rter Identit\u00e4t die Br\u00fccke in ein Daueraufenthaltsrecht bauen wollen."  
example_response_1 = "Genau! So ist es!"  
<br>
example_context_1 = "Sie schaffen dieses Recht nur, weil Sie damit Personen mit ungekl\u00e4rter Identit\u00e4t die Br\u00fccke in ein Daueraufenthaltsrecht bauen wollen.{Alexander Throm, CDU/CSU, Andrea Lindholz, CDU/CSU}"  
example_response_1 = "(Beifall bei der CDU/CSU: Genau! So ist es!"

## Zero-Shot-Learning

In [55]:
# Zero-Shot Prompt

def generate_zero_shot_prompt(preceding_context):
  prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

  {system_prompt}<|eot_id|><|start_header_id|>user<|end_header_id|>

  {preceding_context}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""

  return prompt

Wie erwartet bringt der Prompt unbefriedigende Ergebnisse. Zum Teil wird die Struktur nicht eingehalten, und die Antworten ergeben teilweise keinen Sinn.  
> (Beifall bei So sehen Sie das, ja? von der CDU / CSU:)) -> Struktur
> (Bravo bei der CDU/CSU: Es ist an der Zeit, dass die Regierung ihre Ziele klar benennt!) -> Bravo, aber Unterbrechung passt nicht dazu

Ein besserer Prompt kann möglicherweise mit Zero-Shot-Learning auch befriedigende Ergebnisse liefern.

## One-Shot-Learning

In [95]:
# One-Shot Prompt

def generate_one_shot_prompt(preceding_context):
  prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

  {system_prompt}<|eot_id|><|start_header_id|>user<|end_header_id|>

  {example_context_1}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

  {example_response_1}<|eot_id|>|start_header_id|>user<|end_header_id|>

  {preceding_context}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""

  return prompt

Mein erster One-Shot-Prompt hat überraschenderweise eine sehr gute Unterbrechung geliefert.  
<br>
preceding_context:  
> Ich finde Sie eigentlich
sympathisch. Deswegen tut es mir immer Leid, dass Sie dann,
wenn Sie hier vorne sprechen, langfristiger und
kurzfristiger Gedächtnisschwund befällt.

<br>
Tatsächliche Unterbrechung:  

> Na, na!

<br>
Generierte Antworten:  

> (Zustimmung bei der CDU / CSU: Nein, nein, das muss nicht sein!)  
(Unruhe in den Regalen bei der SPD: Das ist uns zu viel, das war zu weit!)  
(Erschrockener Aufschrei von der Regierungskoalition bei der CDU/CSU: Nein, nein, das ist nicht fair!)  
(Beifall bei der SPD: Das ist ein unangemessener Angriff!)

<br>
Wie man sehen kann, sind die generierten Antworten sehr nahe zu "Na, na!". Z.B. "Nein, nein, das muss nicht sein!"

## Few-Shot-Learning

Es wurden zuerst die ersten vier Beispiele benutzt für das Few-Shot-Learning. Dabei kamen aber im Gegensatz zu meinen Erwartungen auch Unsinn raus.

Tatsächliche Unterbrechung:  
> Frau Präsidentin, ich würde gerne eine Zusatzfrage stellen!  
  
Generated responses:  
> 1: (Zustimmung bei Abgeordneten der SPD: Sehr richtig!)  
2: (Francesca Poggment aus dem Publikum: Wir danken Ihnen, Herr Minister!  - Beifall bei den Abgeordneten der SPD und CDU/CSU )  
3: (Frau Kollegin, ich habe eine Frage: Wie rechnen Sie das Europäische Jahr der Menschen mit Behinderungen und all diese Initiativen in die Umsetzung des Sozialgesetzbuches ein  
4: (Beifall)  

Es wurde z.B. nicht mal eine Unterbrechung generiert.

In [135]:
# Few-Shot-Prompt

def generate_few_shot_prompt(preceding_context):
  prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

  {system_prompt}<|eot_id|><|start_header_id|>user<|end_header_id|>

  {example_context_1}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

  {example_response_1}<|eot_id|>|start_header_id|>user<|end_header_id|>

  {example_context_2}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

  {example_response_2}<|eot_id|>|start_header_id|>user<|end_header_id|>

  {example_context_3}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

  {example_response_3}<|eot_id|>|start_header_id|>user<|end_header_id|>

  {example_context_4}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

  {example_response_4}<|eot_id|>|start_header_id|>user<|end_header_id|>

  {example_context_5}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

  {example_response_5}<|eot_id|>|start_header_id|>user<|end_header_id|>

  {preceding_context}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""

  return prompt

---

Erkenntnis:<br>
(Beifall bei CDU / CSU:) Aber was tut die Regierung dagegen? Wir haben immer wieder versucht, mehr R<br>
<br>
(Beifall bei CDU / CSU:) Aber warum gibt es in den letzten Jahren keine konkreten Maßnahmen von der<br>
<br>
max_tokens (30) war zu niedrig. Erhöht auf 50. (Intention, eher Zurufe zu generieren statt Konversationen)

---

Methode, um den workflow zu starten:

In [162]:
def run(n_prompts, learning_type):
  # Load configurations from the JSON file
  with open('configurations.json', 'r') as f:
    configurations = json.load(f)

  for i in range(n_prompts): # amount of prompts/context
    preceding_context, interruptor, actual_interruption = get_interruption_info(parsed_protocols)
    # OR Write your own speech
    # preceding_context = "MANUAL SPEECH"
    # interruptor = {
        # "name": ""
        # "party": ""
    # }

    try:
      if(learning_type=="zero-shot"):
        prompt = generate_zero_shot_prompt(preceding_context)
      elif(learning_type=="one-shot"):
        prompt = generate_one_shot_prompt(preceding_context)
      elif(learning_type=="few-shot"):
        prompt = generate_few_shot_prompt(preceding_context)
      else:
        raise Exception("Unknown learning type")
    except Exception as e:
      if str(e) == "No protocols with comments found or all have been processed.":
        break
      else:
        raise e

    for config in configurations:
      llm = setup_llm( # seed?
          new_n=n,
          new_n_ctx=config['n_ctx'],
          new_max_tokens=config['max_tokens'],
          new_temperature=config['temperature'],
          new_top_p=config['top_p'],
          new_top_k=config['top_k']
      )

      n_ctx_trunc = 0.7

      while n_ctx_trunc > 0:
        try:
          truncated_prompt, truncated_context = truncate_speech(prompt, preceding_context, n_ctx_trunc)

          responses = generate_responses(truncated_prompt)

          break
        except ValueError as e:
          n_ctx_trunc -= 0.1

      append_interruption(responses)

    save_interruptions(truncated_prompt, preceding_context, interruptor, interruptions, actual_interruption)

In [20]:
parsed_protocols[0]

[{'speaker': {'name': 'Sebastian Hartmann', 'party': 'SPD'},
  'text': 'Sehr geehrte Frau Präsidentin! Liebe Kolleginnen und Kollegen! Wir alle stehen sicherlich unter dem Eindruck der sehr bewegenden, sehr würdigen Gedenkveranstaltung, die gerade eben hier im Plenum stattgefunden hat. Trotzdem möchte ich sagen, dass wir jetzt zu einem anderen inhaltlichen Thema überleiten, dem Wahlrecht. Danke aber noch mal an das Präsidium für die Ausrichtung dieser Gedenkveranstaltung!\n(Beifall bei der SPD, dem GRÜNE und der FDP sowie bei Abgeordneten der CDU/CSU und der LINKEN)\nSehr geehrte Frau Präsidentin, liebe Kolleginnen und Kollegen, mit dem Gesetzentwurf zur Wahlrechtsreform legen wir nicht das x‑te kosmetische Reförmchen vor oder machen den x‑ten kleinen Schritt, sondern wir wagen einen großen Wurf: Wir führen den Deutschen Bundestag zukünftig bei allen weiteren Wahlen auf die Regelgröße von 598 Sitzen zurück.\n(Beifall bei der SPD und dem GRÜNE sowie bei Abgeordneten der FDP)\nDamit löse

In [75]:
run(1, "zero-shot")

llama_model_loader: loaded meta data with 21 key-value pairs and 291 tensors from /root/.cache/huggingface/hub/models--bartowski--Llama-3-SauerkrautLM-8b-Instruct-GGUF/snapshots/b18782be69a9da7120a3fb54f7da765990020cd0/./Llama-3-SauerkrautLM-8b-Instruct-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = Llama-3-SauerkrautLM-8b-Instruct
llama_model_loader: - kv   2:                          llama.block_count u32              = 32
llama_model_loader: - kv   3:                       llama.context_length u32              = 8192
llama_model_loader: - kv   4:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336

Model setup with n_ctx: 4096, temperature: 0.4, top_p: 0.85, top_k: 40



llama_print_timings:        load time =   47722.99 ms
llama_print_timings:      sample time =      48.88 ms /    24 runs   (    2.04 ms per token,   491.05 tokens per second)
llama_print_timings: prompt eval time =   48564.78 ms /   521 tokens (   93.21 ms per token,    10.73 tokens per second)
llama_print_timings:        eval time =    7571.96 ms /    23 runs   (  329.22 ms per token,     3.04 tokens per second)
llama_print_timings:       total time =   56216.46 ms /   544 tokens
llama_model_loader: loaded meta data with 21 key-value pairs and 291 tensors from /root/.cache/huggingface/hub/models--bartowski--Llama-3-SauerkrautLM-8b-Instruct-GGUF/snapshots/b18782be69a9da7120a3fb54f7da765990020cd0/./Llama-3-SauerkrautLM-8b-Instruct-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv

(Beifall bei der AfD: Wir sollten uns an die eigentlichen Ziele erinnern!)


llm_load_vocab: missing pre-tokenizer type, using: 'default'
llm_load_vocab:                                             
llm_load_vocab: ************************************        
llm_load_vocab: GENERATION QUALITY WILL BE DEGRADED!        
llm_load_vocab: CONSIDER REGENERATING THE MODEL             
llm_load_vocab: ************************************        
llm_load_vocab:                                             
llm_load_vocab: special tokens cache size = 256
llm_load_vocab: token to piece cache size = 0.8000 MB
llm_load_print_meta: format           = GGUF V3 (latest)
llm_load_print_meta: arch             = llama
llm_load_print_meta: vocab type       = BPE
llm_load_print_meta: n_vocab          = 128256
llm_load_print_meta: n_merges         = 280147
llm_load_print_meta: n_ctx_train      = 8192
llm_load_print_meta: n_embd           = 4096
llm_load_print_meta: n_head           = 32
llm_load_print_meta: n_head_kv        = 8
llm_load_print_meta: n_layer          = 32
llm_load_pri

...


llama_print_timings:        load time =   46119.02 ms
llama_print_timings:      sample time =     102.64 ms /    49 runs   (    2.09 ms per token,   477.40 tokens per second)
llama_print_timings: prompt eval time =   46590.27 ms /   517 tokens (   90.12 ms per token,    11.10 tokens per second)
llama_print_timings:        eval time =   15237.77 ms /    48 runs   (  317.45 ms per token,     3.15 tokens per second)
llama_print_timings:       total time =   61990.71 ms /   565 tokens
llama_model_loader: loaded meta data with 21 key-value pairs and 291 tensors from /root/.cache/huggingface/hub/models--bartowski--Llama-3-SauerkrautLM-8b-Instruct-GGUF/snapshots/b18782be69a9da7120a3fb54f7da765990020cd0/./Llama-3-SauerkrautLM-8b-Instruct-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv

(Abgelehnt bei der SPD: Warum sollte man den Aussagen von Herrn Kopper so viel Bedeutung beilegen? Es handelt sich um eine private Bank, die ihre Interessen vertreten will.)


llama_model_loader: - kv  16:                      tokenizer.ggml.merges arr[str,280147]  = ["Ġ Ġ", "Ġ ĠĠĠ", "ĠĠ ĠĠ", "...
llama_model_loader: - kv  17:                tokenizer.ggml.bos_token_id u32              = 128000
llama_model_loader: - kv  18:                tokenizer.ggml.eos_token_id u32              = 128009
llama_model_loader: - kv  19:                    tokenizer.chat_template str              = {% set loop_messages = messages %}{% ...
llama_model_loader: - kv  20:               general.quantization_version u32              = 2
llama_model_loader: - type  f32:   65 tensors
llama_model_loader: - type q4_K:  193 tensors
llama_model_loader: - type q6_K:   33 tensors
llm_load_vocab: missing pre-tokenizer type, using: 'default'
llm_load_vocab:                                             
llm_load_vocab: ************************************        
llm_load_vocab: GENERATION QUALITY WILL BE DEGRADED!        
llm_load_vocab: CONSIDER REGENERATING THE MODEL             
llm_load_

Model setup with n_ctx: 4096, temperature: 0.7, top_p: 0.95, top_k: 40


KeyboardInterrupt: 

In [115]:
run(1, "one-shot")

llama_model_loader: loaded meta data with 21 key-value pairs and 291 tensors from /root/.cache/huggingface/hub/models--bartowski--Llama-3-SauerkrautLM-8b-Instruct-GGUF/snapshots/b18782be69a9da7120a3fb54f7da765990020cd0/./Llama-3-SauerkrautLM-8b-Instruct-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = Llama-3-SauerkrautLM-8b-Instruct
llama_model_loader: - kv   2:                          llama.block_count u32              = 32
llama_model_loader: - kv   3:                       llama.context_length u32              = 8192
llama_model_loader: - kv   4:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336

Model setup with n_ctx: 8192, temperature: 0.8, top_p: 0.85, top_k: 40



llama_print_timings:        load time =   42838.07 ms
llama_print_timings:      sample time =      70.33 ms /    36 runs   (    1.95 ms per token,   511.85 tokens per second)
llama_print_timings: prompt eval time =  118353.86 ms /  1224 tokens (   96.69 ms per token,    10.34 tokens per second)
llama_print_timings:        eval time =   11926.38 ms /    35 runs   (  340.75 ms per token,     2.93 tokens per second)
llama_print_timings:       total time =  130399.22 ms /  1259 tokens
llama_model_loader: loaded meta data with 21 key-value pairs and 291 tensors from /root/.cache/huggingface/hub/models--bartowski--Llama-3-SauerkrautLM-8b-Instruct-GGUF/snapshots/b18782be69a9da7120a3fb54f7da765990020cd0/./Llama-3-SauerkrautLM-8b-Instruct-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv

(Beifall bei der CDU / CSU: Das ist ein Schauspiel! Und noch immer redet Herr Steinbrück in den Medien weiter!)


llama_model_loader: - kv  20:               general.quantization_version u32              = 2
llama_model_loader: - type  f32:   65 tensors
llama_model_loader: - type q4_K:  193 tensors
llama_model_loader: - type q6_K:   33 tensors
llm_load_vocab: missing pre-tokenizer type, using: 'default'
llm_load_vocab:                                             
llm_load_vocab: ************************************        
llm_load_vocab: GENERATION QUALITY WILL BE DEGRADED!        
llm_load_vocab: CONSIDER REGENERATING THE MODEL             
llm_load_vocab: ************************************        
llm_load_vocab:                                             
llm_load_vocab: special tokens cache size = 256
llm_load_vocab: token to piece cache size = 0.8000 MB
llm_load_print_meta: format           = GGUF V3 (latest)
llm_load_print_meta: arch             = llama
llm_load_print_meta: vocab type       = BPE
llm_load_print_meta: n_vocab          = 128256
llm_load_print_meta: n_merges         = 28014

...

(Gelächter bei der CDU / CSU: Herr Steinbrück hat es aber auch mal über den Pensionsfonds gesagt, da ging man in die erste Reihe, um vorne sitzen zu können!)


llama_model_loader: - kv  16:                      tokenizer.ggml.merges arr[str,280147]  = ["Ġ Ġ", "Ġ ĠĠĠ", "ĠĠ ĠĠ", "...
llama_model_loader: - kv  17:                tokenizer.ggml.bos_token_id u32              = 128000
llama_model_loader: - kv  18:                tokenizer.ggml.eos_token_id u32              = 128009
llama_model_loader: - kv  19:                    tokenizer.chat_template str              = {% set loop_messages = messages %}{% ...
llama_model_loader: - kv  20:               general.quantization_version u32              = 2
llama_model_loader: - type  f32:   65 tensors
llama_model_loader: - type q4_K:  193 tensors
llama_model_loader: - type q6_K:   33 tensors
llm_load_vocab: missing pre-tokenizer type, using: 'default'
llm_load_vocab:                                             
llm_load_vocab: ************************************        
llm_load_vocab: GENERATION QUALITY WILL BE DEGRADED!        
llm_load_vocab: CONSIDER REGENERATING THE MODEL             
llm_load_

Model setup with n_ctx: 8192, temperature: 0.95, top_p: 0.95, top_k: 40


Llama.generate: prefix-match hit

llama_print_timings:        load time =   42838.07 ms
llama_print_timings:      sample time =      61.14 ms /    31 runs   (    1.97 ms per token,   507.02 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     0 tokens (    -nan ms per token,     -nan tokens per second)
llama_print_timings:        eval time =   10547.49 ms /    31 runs   (  340.24 ms per token,     2.94 tokens per second)
llama_print_timings:       total time =   10646.94 ms /    31 tokens


(Beifall bei CDU / CSU: Herr Steinbrück sollte sich an seine Vergangenheit erinnern!  )
Data saved to text_generation_results.json


---

In [169]:
run(100, "few-shot")

llama_model_loader: loaded meta data with 21 key-value pairs and 291 tensors from /root/.cache/huggingface/hub/models--bartowski--Llama-3-SauerkrautLM-8b-Instruct-GGUF/snapshots/b18782be69a9da7120a3fb54f7da765990020cd0/./Llama-3-SauerkrautLM-8b-Instruct-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = Llama-3-SauerkrautLM-8b-Instruct
llama_model_loader: - kv   2:                          llama.block_count u32              = 32
llama_model_loader: - kv   3:                       llama.context_length u32              = 8192
llama_model_loader: - kv   4:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336

Model setup with n_ctx: 8192, temperature: 0.8, top_p: 0.85, top_k: 40



llama_print_timings:        load time =   42838.07 ms
llama_print_timings:      sample time =      42.17 ms /    20 runs   (    2.11 ms per token,   474.29 tokens per second)
llama_print_timings: prompt eval time =  122643.42 ms /  1240 tokens (   98.91 ms per token,    10.11 tokens per second)
llama_print_timings:        eval time =    6713.42 ms /    19 runs   (  353.34 ms per token,     2.83 tokens per second)
llama_print_timings:       total time =  129427.39 ms /  1259 tokens
Llama.generate: prefix-match hit


(Freie Rufe von den Reihen der SPD und GRÜNE)



llama_print_timings:        load time =   42838.07 ms
llama_print_timings:      sample time =      31.19 ms /    15 runs   (    2.08 ms per token,   480.94 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     0 tokens (    -nan ms per token,     -nan tokens per second)
llama_print_timings:        eval time =    5333.44 ms /    15 runs   (  355.56 ms per token,     2.81 tokens per second)
llama_print_timings:       total time =    5385.50 ms /    15 tokens
llama_model_loader: loaded meta data with 21 key-value pairs and 291 tensors from /root/.cache/huggingface/hub/models--bartowski--Llama-3-SauerkrautLM-8b-Instruct-GGUF/snapshots/b18782be69a9da7120a3fb54f7da765990020cd0/./Llama-3-SauerkrautLM-8b-Instruct-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv

...

(Störung bei der CDU / CSU: Antworthen Sie mal in der Sitzung!)
Data saved to text_generation_results.json


llama_model_loader: - kv   7:              llama.attention.head_count_kv u32              = 8
llama_model_loader: - kv   8:                       llama.rope.freq_base f32              = 500000.000000
llama_model_loader: - kv   9:     llama.attention.layer_norm_rms_epsilon f32              = 0.000010
llama_model_loader: - kv  10:                          general.file_type u32              = 15
llama_model_loader: - kv  11:                           llama.vocab_size u32              = 128256
llama_model_loader: - kv  12:                 llama.rope.dimension_count u32              = 128
llama_model_loader: - kv  13:                       tokenizer.ggml.model str              = gpt2
llama_model_loader: - kv  14:                      tokenizer.ggml.tokens arr[str,128256]  = ["!", "\"", "#", "$", "%", "&", "'", ...
llama_model_loader: - kv  15:                  tokenizer.ggml.token_type arr[i32,128256]  = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...
llama_model_loader: - kv  16:                 

Model setup with n_ctx: 8192, temperature: 0.8, top_p: 0.85, top_k: 40


Llama.generate: prefix-match hit


KeyboardInterrupt: 

---

Ich gebe die Ergebnisse nochmal lesbarer im Notebook aus:

In [170]:
with open('text_generation_results.json', 'r') as f:
    data = json.load(f)

for prompt in data:
  print(prompt['preceding_context'][-300:])
  print()
  try:
    print("Actual interruption:", prompt['actual_interruption'])
  except KeyError:
    print("No actual interruption noted")

  print("\n\nGenerated responses:")
  i = 1
  for interruption in prompt['interruptions']:
    for response in interruption['responses']:
      print(f"{i}:", response)
      i += 1
  print("\n\n")

gierung endlich einen Ruck 
gäbe!) 
— Wir können uns z. B. nicht erlauben, Herr Kollege 
Kolbow, daß bei uns im Lande nach diesem Erfolg der 
Eindruck entsteht: Der Mohr hat seine Schuldigkeit 
getan, er kann gehen. 
(Beifall bei der CDU/CSU und der FDP — 
{Willy Wimmer, CDU/CSU, Walter Kolbow, SPD}

No actual interruption noted


Generated responses:
1: (Beifall bei So sehen Sie das, ja? von der CDU / CSU:))
2: (Beifall bei "Das ist aber ein neues Anliegen Ihrer Partei!" CDU / CSU:)
3: (Beifall bei der AfD: Wir sollten uns an die eigentlichen Ziele erinnern!)
4: (Beifall bei der CDU / CSU: Das war ein wichtiger Punkt, Herr Altherr!)
5: (Beifall bei der FDP: Es ist ja eine klare Kritik an den Sozialdemokraten, dass sie immer erst nach dem Erfolg auftreten.)
6: (Beifall bei der FDP: Ich denke, es ging um die Inhalte, nicht um Personal!)
7: (Beifall bei der CDU / CSU: Es ist ein offener Vorwurf!)
8: (Beifall bei der CDU / CSU: Der Eindruck ist entstanden!)
9: (Beiifall bei der FDP: Es so

---

## Evaluation
---

Da es schwierig ist, diese Text Generation zu evaluieren, muss ich selber auf die Ergebnisse schauen, um zu prüfen, ob die Ergebnisse Sinn ergeben.  
Dabei kommen auch gute Ergebnisse raus:

> Actual interruption: Schämen Sie sich!  
> Generated interruption: (Beifall bei der LINKE: Schämen Sie sich!)  


> Actual interruption: Richtig!  
> Generated interruption: (Zustimmung bei der CDU/CSU: Ja!)  


> Actual interruption: Das ist ein Witz!  
> Generated interruption: (Beifall bei der LINKE: Das ist ein verlogener Vorwurf!)  

<br>

Aber auch falsche:  
> Actual interruption: Dass Sie keinen Plan haben, glaube ich!  
> Generated interruption: (Beifall bei der GRÜNE: Sehr gut!)  


> Actual interruption: Na! Na! Na!  
> Generated interruption: (Beifall bei der FDP: Bravo!)  

<br>

Manchmal werden auch gar keine Antworten generiert:  
> Generated interruption: (Abbruch des Beifalls)  
> Generated interruption: (Beifall bei den Regierungsparteien)   
> Generated interruption: (Fragen von Wolfgang Götze, DIE LINKE)  
> Generated interruption: (Bell)  

Für die komplette Evaluation vergleich man die Ergebnisse in [text_generation_results](text_generation_results.json)


Mit weiterem Aufwand könnte man wahrscheinlich die fehlerhaften/leeren Antworten reduzieren. Auch hier bin ich wegen zeitlichen Gründen nicht dazu gekommen.

Ein weiteres Beispiel, wo die Antwort am Ende Unsinn ausgegeben hat



In [ ]:
print_response(response)



(Einwurf von Alexander Gallus [Linke]: Was ist mit der Datenlage, die an der Stelle fehlt? Wo sind die Zahlen über die Strafverfolgung und -ahndigung? – R


# Notizen

Geplant war es noch, mehr mit max_tokens, n_ctx, temperature, top_p, top_k und den Prompts / Beispielen auszutesten.  
<br>
Außerdem sollten die Unterbrechungen aus [IV_interruption_prediction](../IV_interruption_prediction/) Unterbrechungszeitpunkte liefern, um dann daraus Unterbrechungen zu generieren.  


Für das Few-Shot-Prompting sollten ebenso mehr Tests durchgeführt werden.  
Meine Ansätze wären zum Beispiel gewesen:  


*   Mehr oder weniger Beispiele zum Prompt hinzufügen
*   System Prompt verändern
*   n_ctx / Kontextgröße verringern, um zu testen, ob das Modell mit weniger Kontext auch gute Unterbrechungen generiert
*   Genug Beispiele hinzufügen, sodass die Unterbrechung an den Unterbrecher angepasst wird, um dann Personen zu imitieren
